# Wavelet-CLIP Deepfake & AI Video Detector Demo

A comprehensive notebook for setting up and using Wavelet-CLIP, a state-of-the-art deepfake and AI video detection system. Includes model loading, video processing, detection inference, and identity verification.

**Sections:**
1. Environment Setup and Repository Clone
2. Install Dependencies
3. Download and Configure Pretrained Models
4. Dataset Configuration
5. Load Wavelet-CLIP Model and CLIP
6. Implement Wavelet Feature Extraction
7. Single Video Detection Function
8. Batch Video Processing
9. Visualize Detection Results
10. Bonus: FaceNet Identity Verification
11. Performance Benchmarking

## 1. Environment Setup and Repository Clone

Clone the official Wavelet-CLIP repository and navigate to the project directory.

In [ ]:
# Clone the official Wavelet-CLIP repository
!git clone https://github.com/lalithbharadwajbaru/wavelet-clip.git
%cd wavelet-clip

## 2. Install Dependencies

Install all required Python packages including PyTorch with CUDA support, timm, OpenCV, scikit-learn, tqdm, and OpenAI CLIP.

In [ ]:
# Install dependencies (run in shell)
!sh install.sh
# If install.sh fails, run the following:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install timm opencv-python scikit-learn tqdm
!pip install git+https://github.com/openai/CLIP.git
!pip install facenet-pytorch

## 3. Download and Configure Pretrained Models

Download the pretrained Wavelet-CLIP weights (e.g., `clip_wavelet_best.pth`) and place them in `./training/weights/`. Confirm the weight path in `./training/config/detector/detector.yaml`.

**Instructions:**
- Download the latest weights from the Wavelet-CLIP GitHub releases or Google Drive link in the repo.
- Place the file (e.g., `clip_wavelet_best.pth`) in `training/weights/`.
- Edit `training/config/detector/detector.yaml` to set the correct weights path.

## 4. Dataset Configuration

Set up dataset paths in `dataset.yaml` for both benchmark datasets (FaceForensics++, Celeb-DF) and your own videos.

In [ ]:
# Example: Add your custom dataset to dataset.yaml
# In training/config/dataset/dataset.yaml, add:
#
# MYDATA:
#   path: "/full/path/to/my_videos/"
#   type: video

# Place your .mp4 or .avi files in the my_videos/ directory.

## 5. Load Wavelet-CLIP Model and CLIP

Import necessary modules, load the pretrained CLIP model (ViT-L/14), and initialize the WaveletCLIP classifier with trained weights.

In [ ]:
import torch
import clip
from wavelet_utils import apply_wavelet_transform  # Provided in repo
from model import WaveletCLIP  # Provided in repo
from PIL import Image

# Load CLIP
clip_model, preprocess = clip.load("ViT-L/14", device="cuda")
# Load Trained Wavelet-CLIP Classifier
model = WaveletCLIP().cuda()
model.load_state_dict(torch.load("training/weights/clip_wavelet_best.pth"))
model.eval()

## 6. Implement Wavelet Feature Extraction

Create a function to extract frames from videos, apply wavelet transforms, and preprocess them into feature tensors suitable for the model.

In [ ]:
import cv2

def video_to_wavelet_features(video_path, max_frames=16):
    cap = cv2.VideoCapture(video_path)
    frames = []
    for _ in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = preprocess(Image.fromarray(img)).unsqueeze(0).cuda()
        wavelet_features = apply_wavelet_transform(img)  # From repo
        frames.append(wavelet_features)
    cap.release()
    return torch.cat(frames, dim=0)

## 7. Single Video Detection Function

Implement a complete detection pipeline that processes a single video file and returns the fakeness probability using the Wavelet-CLIP model.

In [ ]:
def detect_fake_video(video_path):
    features = video_to_wavelet_features(video_path)
    with torch.no_grad():
        logits = model(features)
        prob_fake = torch.softmax(logits, dim=1)[:, 1].mean().item()
    print(f"Video '{video_path}': Fakeness probability = {prob_fake:.3f}")
    return prob_fake

# Example usage:
detect_fake_video("my_videos/test.mp4")

## 8. Batch Video Processing

Create a batch processing function to analyze multiple videos from a directory and aggregate detection results into a DataFrame.

In [ ]:
import os
import pandas as pd

def batch_detect_videos(video_dir):
    results = []
    for fname in os.listdir(video_dir):
        if fname.endswith('.mp4') or fname.endswith('.avi'):
            path = os.path.join(video_dir, fname)
            prob_fake = detect_fake_video(path)
            results.append({'video': fname, 'fakeness': prob_fake})
    df = pd.DataFrame(results)
    return df

# Example usage:
# df_results = batch_detect_videos('my_videos/')
# print(df_results)

## 9. Visualize Detection Results

Use matplotlib to create bar charts and confusion matrices showing detection probabilities and classification accuracy across test videos.

In [ ]:
import matplotlib.pyplot as plt

def plot_detection_results(df):
    plt.figure(figsize=(10,6))
    plt.bar(df['video'], df['fakeness'])
    plt.ylabel('Fakeness Probability')
    plt.xlabel('Video')
    plt.title('Wavelet-CLIP Detection Results')
    plt.xticks(rotation=45)
    plt.show()

# Example usage:
# plot_detection_results(df_results)

## 10. Bonus: FaceNet Identity Verification

Integrate FaceNet (InceptionResnetV1) to extract face embeddings from videos and implement identity verification by comparing embeddings using cosine similarity.

In [ ]:
from facenet_pytorch import InceptionResnetV1
from PIL import Image
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

facenet_model = InceptionResnetV1(pretrained='vggface2').eval().cuda()

def get_embed(image_path):
    img = preprocess(Image.open(image_path)).unsqueeze(0).cuda()
    emb = facenet_model(img)
    return emb.cpu().detach().numpy()[0]

def compare_embeddings(emb1, emb2):
    return cosine_similarity([emb1], [emb2])[0][0]

# Example usage:
# emb_ref = get_embed('reference.jpg')
# emb_video = get_embed('frame_from_video.jpg')
# similarity = compare_embeddings(emb_ref, emb_video)
# print(f'Cosine similarity: {similarity:.3f}')

## 11. Performance Benchmarking

Run the detector on standard benchmark datasets, calculate metrics (accuracy, precision, recall, F1-score), and compare results against published baselines.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def benchmark_results(df, true_labels):
    preds = (df['fakeness'] > 0.5).astype(int)
    acc = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds)
    rec = recall_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)
    print(f"Accuracy: {acc:.3f}\nPrecision: {prec:.3f}\nRecall: {rec:.3f}\nF1-score: {f1:.3f}")
    return acc, prec, rec, f1

# Example usage:
# true_labels = [0, 1, 0, 1]  # 0=real, 1=fake
# benchmark_results(df_results, true_labels)